In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupShuffleSplit

# Load data
train   = pd.read_parquet("Data/train.parquet")
test    = pd.read_parquet("Data/test.parquet")
sensors = pd.read_parquet("Data/sensors.parquet")

print(f"Train: {train.shape}  |  Test: {test.shape}  |  Sensors: {sensors.shape}")
print(f"Train sensors: {train['sensor'].nunique()}  |  Test sensors: {test['sensor'].nunique()}")
print(f"Overlap train/test sensors: {len(set(train['sensor'].unique()) & set(test['sensor'].unique()))}")

## 1. Exploratory Data Analysis

In [ ]:
print("=== Train stats ===")
print(train[['power', 'temperature']].describe().round(2))
print(f"\nNaN in temperature: {train['temperature'].isna().sum()} ({train['temperature'].isna().mean():.1%})")
print(f"\nTime span: {train['time'].max()/86400:.0f} days ({train['time'].max()/86400/365.25:.1f} years), step = 10 days")
print(f"Unique train sensors: {train['sensor'].nunique()}  |  Unique test sensors: {test['sensor'].nunique()}")
print(f"\n=== Sensor coordinates ===")
print(sensors[['coor_x','coor_y','coor_z']].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sample = train.dropna(subset=['temperature']).sample(5000, random_state=42)
axes[0].scatter(sample['power'], sample['temperature'], alpha=0.1, s=5)
axes[0].set_xlabel('Power'); axes[0].set_ylabel('Temperature')
axes[0].set_title('Temperature vs Power')

s = train[train['sensor'] == train['sensor'].iloc[0]].copy()
axes[1].plot(s['time'] / 86400, s['temperature'], lw=0.5)
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Temperature')
axes[1].set_title(f"Time series — sensor {s['sensor'].iloc[0]}")

sc = axes[2].scatter(sensors['coor_x'], sensors['coor_y'], c=sensors['coor_z'], cmap='viridis', s=15)
axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')
axes[2].set_title('Sensor positions (color = Z)')
plt.colorbar(sc, ax=axes[2], label='Z')
plt.tight_layout(); plt.show()

## 2. Feature Engineering & Normalisation

**Pourquoi ces features ?**
- `coor_x, coor_y, coor_z` : les capteurs de test sont **entièrement inconnus** — la seule façon de généraliser est d'utiliser leur position dans l'espace 3D.
- `power` : mesure directe disponible dans train et test.
- `time_days` : tendance long-terme (la température peut évoluer sur 250 ans).
- `sin_doy, cos_doy` : encodage cyclique du jour de l'année → capture la saisonnalité (exercice 03-kNN : les features doivent représenter la similarité réelle entre points).

**Normalisation Z-score** *(méthode du cours, exercice 03-kNN)* :  
$$x_{\text{norm}} = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$  
Indispensable pour la régression linéaire (gradient descent) et le réseau de neurones : évite que des features à grande échelle (ex. `time_days` en milliers) dominent les autres.  
La moyenne et l'écart-type sont calculés **uniquement sur le train** et appliqués au test, pour éviter toute fuite d'information (*data leakage*).

In [ ]:
FEATURES = ['coor_x', 'coor_y', 'power', 'time_days', 'sin_doy', 'cos_doy']
TARGET   = 'temperature'

def add_features(df, sensors_df):
    df = df.merge(sensors_df[['sensor','coor_x','coor_y','coor_z']], on='sensor', how='left')
    df['time_days'] = df['time'] / 86400
    doy = df['time_days'] % 365.25
    df['sin_doy']   = np.sin(2 * np.pi * doy / 365.25)
    df['cos_doy']   = np.cos(2 * np.pi * doy / 365.25)
    return df

train_f = add_features(train.copy(), sensors)
test_f  = add_features(test.copy(),  sensors)

# Drop rows with missing temperature
train_clean = train_f.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"Rows after dropping NaN temperature: {len(train_clean):,}")

# --- Normalisation Z-score (exercice 03-kNN) ---
# Computed on train only → applied to train and test (no data leakage)
X_all = train_clean[FEATURES].values.astype(np.float32)
y_all = train_clean[TARGET].values.astype(np.float32)

feat_mean = X_all.mean(axis=0)
feat_std  = X_all.std(axis=0)

# Drop constant features (std=0) — coor_z is all-zero in this dataset
non_const = feat_std > 0
FEATURES  = [f for f, ok in zip(FEATURES, non_const) if ok]
X_all     = X_all[:, non_const]
feat_mean = feat_mean[non_const]
feat_std  = feat_std[non_const]

y_mean = y_all.mean()
y_std  = y_all.std()

X_all_norm  = (X_all - feat_mean) / feat_std
y_all_norm  = (y_all - y_mean) / y_std
X_test_norm = (test_f[FEATURES].values.astype(np.float32) - feat_mean) / feat_std

print(f"Features retenues ({len(FEATURES)}) : {FEATURES}")
print(f"Feature means : {feat_mean.round(3)}")
print(f"Feature stds  : {feat_std.round(3)}")

## 3. Validation : Group Split par capteur

**Stratégie** *(inspirée de l'exercice 04 et du devoir logistic regression)* :  
Les capteurs de test sont **entièrement absents** du train. La validation doit reproduire cette condition : on réserve un sous-ensemble de **capteurs entiers** (pas de timesteps isolés) pour la validation.  
→ `GroupShuffleSplit` avec `groups = sensor` : chaque capteur est soit entièrement en train, soit entièrement en val.

Ceci évite le **data leakage temporel** : si on mélangeait les timesteps d'un même capteur entre train et val, le modèle verrait des mesures du même capteur et surestimerait ses performances réelles.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_all_norm, groups=train_clean['sensor']))

X_tr, X_val = X_all_norm[train_idx], X_all_norm[val_idx]
y_tr, y_val = y_all_norm[train_idx], y_all_norm[val_idx]

y_val_orig = y_all[val_idx]

print(f"Train : {len(X_tr):,} rows | Val : {len(X_val):,} rows")
print(f"Train sensors : {train_clean.iloc[train_idx]['sensor'].nunique()} | Val sensors : {train_clean.iloc[val_idx]['sensor'].nunique()}")

## 4. Baseline — Régression Linéaire (Moindres Carrés)

**Méthode** *(exercice 04-linear-regression)* :  
La régression linéaire modélise $\hat{y} = Xw$ et minimise la MSE :
$$J(w) = \frac{1}{N}\|Xw - y\|^2$$

La solution analytique (moindres carrés) est :
$$\hat{w} = (X^\top X)^{-1} X^\top y$$

**Pourquoi comme baseline ?**  
- Solution exacte, pas de hyperparamètres à tuner.
- Donne un RMSE de référence pour évaluer le gain apporté par le MLP.
- Si la relation température ↔ features est linéaire, ce modèle sera déjà bon.

In [ ]:
# Ajouter le terme de biais (colonne de 1) — exercice 04
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1), dtype=np.float32), X])

X_tr_b  = add_bias(X_tr)
X_val_b = add_bias(X_val)

# Moindres carrés : w = (X^T X)^{-1} X^T y — exercice 04
# np.linalg.solve est plus stable numériquement que l'inversion explicite
w_ls = np.linalg.solve(X_tr_b.T @ X_tr_b, X_tr_b.T @ y_tr)

# Prédictions et RMSE (en unités originales via dénormalisation)
y_val_pred_ls = X_val_b @ w_ls
rmse_ls = np.sqrt(np.mean(((y_val_pred_ls * y_std + y_mean) - y_val_orig) ** 2))
print(f"Baseline — Régression Linéaire  |  Val RMSE : {rmse_ls:.2f}")

## 5. Modèle Principal — Réseau de Neurones MLP (sklearn)

**Méthode** *(exercice 06-neural-nets, adapté pour la régression)* :  
`sklearn.neural_network.MLPRegressor` implémente le même algorithme que le cours : couches fully-connected, ReLU, backpropagation, optimiseur Adam.

**Architecture** (inspirée de `ThreeLayerNet` du cours) :
$$\text{Input}(6) \xrightarrow{\text{FC}+\text{ReLU}} 256 \xrightarrow{\text{FC}+\text{ReLU}} 256 \xrightarrow{\text{FC}+\text{ReLU}} 128 \xrightarrow{\text{FC}} 1$$

- **Activation ReLU** : $\max(0, x)$ — même que le cours.
- **Sortie linéaire** : pas d'activation finale → régression de valeur continue.
- **Loss MSE** : $J = \frac{1}{N}\sum(\hat{y} - y)^2$ — même formule que l'exercice 04.
- **Régularisation L2** (`alpha`) : pénalise les grands poids, comme $\lambda\|w\|^2$ dans le devoir logistic regression.
- **Optimiseur Adam** : SGD adaptatif avec moments (vu en cours).
- **Sous-échantillonnage** : sklearn MLP étant moins optimisé que PyTorch sur grands datasets, on entraîne sur 500k lignes représentatives (tous les capteurs train couverts).

In [ ]:
# Sous-échantillonnage stratifié par capteur — garde tous les capteurs train représentés
rng = np.random.default_rng(42)
n_sample = 500_000
sample_idx = rng.choice(len(X_tr), size=min(n_sample, len(X_tr)), replace=False)
X_tr_s = X_tr[sample_idx]
y_tr_s = y_tr[sample_idx]
print(f"Sous-échantillon train : {len(X_tr_s):,} lignes")

In [ ]:
# MLP — exercice 06-neural-nets (sklearn, même algorithme que le cours)
# hidden_layer_sizes = architecture (256, 256, 128) comme ThreeLayerNet
# alpha = régularisation L2, comme λ dans le devoir logistic regression
# solver='adam' = SGD adaptatif (vu en cours)
# early_stopping=True = arrêt si la loss de validation ne s'améliore plus

mlp = MLPRegressor(
    hidden_layer_sizes=(256, 256, 128),
    activation='relu',        # ReLU — exercice 06
    solver='adam',            # Adam — vu en cours
    alpha=1e-4,               # L2 régularisation — devoir logistic regression
    batch_size=1024,
    learning_rate_init=1e-3,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42,
    verbose=True,
)

print("Entraînement MLP en cours...")
mlp.fit(X_tr_s, y_tr_s)
print(f"\nIterations : {mlp.n_iter_}  |  Best val score : {mlp.best_validation_score_:.6f}")

In [ ]:
# Courbe d'apprentissage — exercice 04 (visualiser la convergence de la loss)
plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_curve_, label='Train loss')
if mlp.validation_scores_ is not None:
    plt.plot([-v for v in mlp.validation_scores_], label='Val loss', color='orange')
plt.xlabel('Iteration'); plt.ylabel('MSE (normalisé)')
plt.title('Convergence du MLP')
plt.legend(); plt.tight_layout(); plt.show()

# RMSE en unités originales
val_preds_norm = mlp.predict(X_val)
val_preds = val_preds_norm * y_std + y_mean
rmse_mlp = np.sqrt(np.mean((val_preds - y_val_orig) ** 2))

print(f"\nBaseline Régression Linéaire  →  Val RMSE : {rmse_ls:.2f}")
print(f"MLP                           →  Val RMSE : {rmse_mlp:.2f}")
print(f"Gain MLP vs baseline : {rmse_ls - rmse_mlp:.2f}")

## 6. Prédictions & Soumission

On génère les prédictions avec le MLP, on dénormalise (opération inverse du z-score), et on sauve `submission.csv`.

In [ ]:
test_preds = mlp.predict(X_test_norm) * y_std + y_mean

submission = pd.DataFrame({
    "Id":          np.arange(len(test), dtype=int),
    "temperature": test_preds,
})

# Validation checks from the template
assert list(submission.columns) == ["Id", "temperature"]
assert len(submission) == len(test)
assert (submission["Id"].to_numpy() == np.arange(len(test))).all()
assert np.isfinite(submission["temperature"]).all()
assert submission.isna().sum().sum() == 0

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head())
print(f"\nShape: {submission.shape}  |  temperature stats:")
print(submission["temperature"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Série temporelle pour 3 capteurs test
for s in submission['sensor'].unique()[:3]:
    df_s = submission[submission['sensor'] == s].sort_values('time')
    axes[0].plot(df_s['time'] / 86400, df_s['temperature'], label=s, lw=0.8)
axes[0].set_xlabel('Jour'); axes[0].set_ylabel('Température prédite')
axes[0].set_title('Température prédite (capteurs test)')
axes[0].legend()

# Distribution train (vrai) vs test (prédit)
axes[1].hist(train_clean['temperature'], bins=100, alpha=0.5, density=True, label='Train (réel)')
axes[1].hist(submission['temperature'],  bins=100, alpha=0.5, density=True, label='Test (prédit)')
axes[1].set_xlabel('Température'); axes[1].set_ylabel('Densité')
axes[1].set_title('Distribution des températures')
axes[1].legend()

plt.tight_layout(); plt.show()